<center><img src="./img/pong-thumbnail.png" width="200" alt="Skills Network Logo"  /></center>
  


## Bugs & troubles:


## Dockers:

- ### Se corrompe la ruta al borrar los directorios en el host
    - Cuando levantamos los dockers a traves del archivo `Makefile` se crea el directorio `data`en el Host. Ese es el directorio donde se crean los volumenes para almacenar los datos persistentes de los volúmenes.
    - En el Makefile tenemos la opción `make clean` que borra los directorios de host para emular un borrrado accidental o intencionado.
    - Al levantar de nuevo los contenedores Docker devuelve este error:

        ```swift
        Error response from daemon: failed to copy files: failed to open target /var/lib/docker/volumes/frontend_data/_data/tsconfig.json: open /var/lib/docker/volumes/frontend_data/_data/tsconfig.json: no such file or directory
        make: *** [all] Error 1
        ````
    - En el `Makeflie`antes de lavantar los contededores revisamos si el directorio para los volumenes en el Host está creado y si no lo está, se crea.
    - Revisamos y vemos que los directorios se crean perfectamente pero Docker no los encuentra.

### Solución 1:

Parece que el problema está relacionado con la forma en que Docker maneja los volúmenes cuando eliminan los directorios del host y luego volvemos a montar los volúmenes. Esto puede causar problemas si Docker todavía tiene referencias a esos volúmenes, incluso después de haber eliminado los archivos físicamente.

***Referencia a volúmenes corruptos o huérfanos:*** Docker puede mantener referencias a volúmenes que no han sido desmontados correctamente, lo que lleva a problemas al tratar de montarlos de nuevo.

***Eliminación de volúmenes mientras están en uso:*** Si eliminamos los archivos en el host directamente, pero Docker aún está utilizando esos volúmenes, puede haber corrupción o fallos en el proceso de re-montaje.

- ### Solución sugerida antes de incluir en el Makefile:
    - Borrar la caché.

In [ ]:
docker system prune -af

### Explicación:
Limpia la caché de Docker, incluyendo contenedores parados, imágenes sin utilizar, y redes no referenciadas. Esta opción es más rápida que un reinicio y podría solucionar el problema

- La solución ```NO FUNCIONA``` sigue lanzando el mismo error cuando levantamos los contenedores.

### Solución 2 (Buena) para incluir en el Makefile:
- Reiniciar Docker

Explicamos en detalle cómo funciona Docker en relación a los volúmenes, qué puede estar causando el problema y por qué reiniciar Docker lo soluciona.:
- ### 1. ¿Qué son los volúmenes de Docker?
    Los volúmenes en Docker son áreas de almacenamiento persistente que permiten que los datos generados y utilizados por los contenedores se mantengan, incluso si los contenedores se eliminan o reinician. Son independientes del ciclo de vida de los contenedores y, generalmente, se almacenan fuera del sistema de archivos del contenedor.

    - Los volúmenes se pueden montar en rutas específicas dentro del sistema de archivos del contenedor.
    - El contenido del volumen persiste incluso si el contenedor se elimina o se recrea.
    - Los volúmenes suelen residir en una carpeta especial en el host que Docker gestiona (como /var/lib/docker/volumes en Linux o una ruta similar en macOS).
- ### 2. ¿Qué ocurre cuando eliminamos manualmente un directorio en el host una vez levantado por primera vez Dockers?
    En el proyecto, montamos directorios del host (por ejemplo, /Users/usuario/data) en contenedores Docker usando volúmenes. Cuando decidimos eliminar manualmente los directorios del host (por ejemplo, usando rm -rf), estmoa eliminando archivos que Docker espera que existan. Sin embargo:

    - **Docker aún tiene referencias a esos volúmenes en su caché** o en su base de datos interna. Aunque hayamos eliminado los archivos físicamente del host, Docker no ha sido informado de ese cambio.
    - **Docker intenta montar ese directorio eliminado** en el contenedor cuando levantamos nuevamente el contenedor, lo que provoca que se corrompa la referencia, ya que los archivos o directorios que espera ya no están.
- ### 3. ¿Por qué ocurre el error?
    El error que obtemos:

In [ ]:
Error response from daemon: failed to copy files: failed to open target /var/lib/docker/volumes/frontend_data/_data/tsconfig.json: open /var/lib/docker/volumes/frontend_data/_data/tsconfig.json: no such file or directory

Este error ocurre porque Docker intenta acceder a un archivo (en este caso `tsconfig.json`) dentro del volumen, pero ya no existe en la ruta del host que está montada. Docker está buscando ese archivo porque aún tiene metadatos y referencias internas que indican que ese archivo debería estar presente.

Esto puede ocurrir porque Docker mantiene una caché o metadatos sobre los volúmenes, y al eliminar el contenido directamente del host sin notificar a Docker, la referencia a esos archivos no se elimina correctamente en la memoria interna de Docker.


- ### 4. ¿Por qué reiniciar Docker lo soluciona?
    Cuando reiniciamos Docker, todos los procesos de Docker se detienen y se reinician, lo que incluye la limpieza de la caché interna y la actualización de los metadatos que Docker tiene sobre sus volúmenes. Básicamente:

    - Al reiniciar Docker, se refrescan las referencias que Docker tiene hacia los volúmenes.
    - Docker vuelve a verificar el estado actual de los volúmenes en el sistema de archivos del host.
    - Si los directorios o archivos ya no existen en el host, Docker ya no los tratará como válidos y evitará intentar montarlos de nuevo, eliminando la causa del error.
    
    Es decir, Docker "se da cuenta" de que los archivos que esperaba ya no están y actúa de acuerdo a esa nueva realidad, en lugar de confiar en una referencia antigua o caché que se ha quedado obsoleta.

- ### 5. ¿Qué está provocando que se corrompa la ruta?
    La ruta se corrompe porque, al eliminar los archivos manualmente desde el host mientras Docker aún los está usando o tiene referencias a ellos, Docker no tiene forma de saber que esos archivos ya no están. Como Docker sigue pensando que los archivos están presentes, y cuando intenta acceder a ellos (en nuestro caso, `tsconfig.json` dentro del volumen), el sistema devuelve un error de **"archivo no encontrado"**.

    Docker depende de la sincronización entre el estado del volumen en el host y lo que él cree que existe en ese volumen. Al eliminar manualmente esos archivos o directorios sin una intervención adecuada de Docker (como detener los contenedores y eliminar los volúmenes correctamente), introduces una discrepancia entre lo que Docker cree y lo que realmente está en el sistema de archivos del host. Esto es lo que provoca la corrupción o el fallo en la referencia.

- ### 6. ¿Cómo solucionar o mitigar este problema?
    Existen varias formas de mitigar este problema para que no tengas que reiniciar Docker cada vez:
    - **Verificar** la existencia de los archivos antes de montar los volúmenes: En el Makefile, vamos hacer una verificación condicional de que los archivos o directorios necesarios existan antes de intentar levantar los contenedores.

    - **Reiniciar Docker automáticamente** cuando se detecte un problema: Si este problema sigue ocurriendo con frecuencia, vamos a automatizar el reinicio de Docker en el Makefile. Este reinicio asegura que Docker refresque sus referencias antes de intentar montar los volúmenes de nuevo.

    - **Incluir** eliminar los volúmenes de 2 maneras: En lugar de eliminar manualmente emulando un borrado accidental o intencionado los archivos del host, vamos a usar dos comandos:
        - clean: Emula el hackeo.
        - delete: Para eliminar los volumenes de Docker controladamente con `docker volume rm`. Nos aseguramos de que Docker actualiza sus referencias internas. Esto mantendría Docker "al tanto" de que esos archivos ya no existen.
- ### 7. ¿Por qué limpiar la caché no ayuda?
    Limpiar la caché no resuelve el problema porque el problema no está en los datos en sí que Docker está almacenando temporalmente en la caché. El problema radica en las referencias internas que Docker mantiene sobre los volúmenes. Al eliminar archivos o directorios directamente desde el host sin que Docker participe en esa eliminación, las referencias se corrompen. Limpiar la caché de Docker solo afecta los datos temporales, no las referencias internas a los volúmenes.

In [ ]:
all: restart_if_needed setup
	@docker compose -f ./src/docker-compose.yml up -d --build

kill_docker:
	@./script/kill_docker.sh
	@open /Applications/Docker.app

restart_if_needed:
	@if [ ! -d "/Users/usuario/data" ]; then \
		echo "Directory /Users/usuario/data not found. Checking Docker status..."; \
		if docker ps -q > /dev/null; then \
			echo "Docker is running. Stopping Docker..."; \
			$(MAKE) kill_docker; \
		else \
			echo "Docker is not running. No need to stop Docker."; \
		fi; \
		if uname -s | grep -i darwin > /dev/null; then \
			echo "Running on macOS. Starting Docker..."; \
			open /Applications/Docker.app; \
		elif uname -s | grep -i linux > /dev/null; then \
			echo "Running on Linux. Starting Docker..."; \
			sudo systemctl start docker; \
		fi; \
		echo "Waiting for Docker to start..."; \
		sleep 10; \
		while ! docker ps > /dev/null 2>&1; do \
			echo "Waiting for Docker to be ready..."; \
			sleep 5; \
		done; \
		echo "Docker is ready."; \
	elif ! docker ps -q > /dev/null; then \
		echo "Docker is not running. Starting Docker..."; \
		if uname -s | grep -i darwin > /dev/null; then \
			echo "Running on macOS. Starting Docker..."; \
			open /Applications/Docker.app; \
		elif uname -s | grep -i linux > /dev/null; then \
			echo "Running on Linux. Starting Docker..."; \
			sudo systemctl start docker; \
		fi; \
		echo "Waiting for Docker to start..."; \
		sleep 10; \
		while ! docker ps > /dev/null 2>&1; do \
			echo "Waiting for Docker to be ready..."; \
			sleep 5; \
		done; \
		echo "Docker is ready."; \
	else \
		echo "Directory /Users/usuario/data exists. No need to restart Docker."; \
	fi


down:
	@docker compose -f ./src/docker-compose.yml down -v

clean:
	sudo rm -rf /Users/usuario/data/sqlite/*
	sudo rm -rf /Users/usuario/data/app/*
	sudo rm -rf /Users/usuario/data/php/*
	sudo rm -rf /Users/usuario/data/frontend/*
	sudo rm -rf /Users/usuario/data/blockchain/*
	sudo rm -rf /Users/usuario/data/security/*
	sudo rm -rf /Users/usuario/data
	@if docker ps -qa | grep -q .; then docker stop $$(docker ps -qa); fi
	@if docker ps -qa | grep -q .; then docker rm $$(docker ps -qa); fi
	@if docker images -qa | grep -q .; then docker rmi $$(docker images -qa); fi
	@if docker volume ls -q | grep -q .; then docker volume rm $$(docker volume ls -q); fi
	@if docker network ls --filter name=transcendence -q | grep -q .; then docker network rm transcendence; fi

setup:
	@mkdir -p /Users/usuario/data
	@mkdir -p /Users/usuario/data/sqlite
	@mkdir -p /Users/usuario/data/app
	@mkdir -p /Users/usuario/data/php
	@mkdir -p /Users/usuario/data/frontend
	@mkdir -p /Users/usuario/data/blockchain
	@mkdir -p /Users/usuario/data/security

delete:
	@docker compose -f ./src/docker-compose.yml down -v
	@if docker volume ls -qf "name=transcendence" | grep -q .; then \
		docker volume rm $$(docker volume ls -qf "name=transcendence"); \
	else \
		echo "No transcendence volumes to remove."; \
	fi

logs:
	@docker compose -f ./src/docker-compose.yml logs -f

.PHONY: all down clean setup delete logs

- ### Errores tenidos en cuenta en el Makefiel:
 - **`make`** : 
    - NO está creado el directorio de volumenes y NO corre Docker:
        - Corre Docker, crea directorio y levanta contenedores.
    - SI está creado el directorio de volumenes y NO corre Docker:
        - Corre Docker y levanta contenedores.
    - NO está creado el directorio de volumenes y SI corre Docker (borrado o hackeo):
        - Cierra Docker, Reinicia Docker y levanta contenedores para evitar perdidas de referencias.
    - Sirve para Linux y para Mac.
- **`make down`** :
    - Tumba los contenedores pero mantiene los directorios de los volumenes en el host
- **`make clean`** :
    - Elimina los directorios del host (emula borrado intencionado o por error)
- **`make delete`** :
    - Borra los volumenes de forma controlada.
- **`make logs`** :
    - Busca errores en los contenedores lavantados.
    

***
***


##  SQLite (Base de datos):

- ### No se ha creado la base de datos.
    - Listamos las tablas:
    ```shell
    docker exec -it sqlite sh
    /var/lib/sqlite # ls
    /var/lib/sqlite
    ```

No vemos nada y ejecutamos `make ps` para saber si el conteneder está en ejecución:

Vemos que el contenedor sqlite SI está en ejecución (lo cual es positivo), el siguiente paso es verificar la base de datos y asegurarnos de que el archivo db.sqlite esté disponible en el contenedor. 

In [ ]:
NAME         IMAGE                                COMMAND                  SERVICE     CREATED              STATUS                                 PORTS
app          src-backend                          "docker-entrypoint.s…"   backend     About a minute ago   Up About a minute                      0.0.0.0:3000->3000/tcp, 8080/tcp
blockchain   avaplatform/avalanchego:latest       "/avalanchego/avalan…"   avalanche   About a minute ago   Exited (1) About a minute ago          
frontend     node:18-alpine                       "docker-entrypoint.s…"   frontend    About a minute ago   Up About a minute                      0.0.0.0:8080->8080/tcp
php          php:8.1-fpm                          "docker-php-entrypoi…"   php         About a minute ago   Up About a minute                      9000/tcp
security     softwaresecurityproject/zap-stable   "sh -c '/zap/zap.sh …"   security    About a minute ago   Up About a minute (health: starting)   0.0.0.0:8081->8081/tcp
sqlite       nouchka/sqlite3                      "tail -f /dev/null"      sqlite      About a minute ago   Up About a minute           

- 1. Accede al contenedor SQLite:

Para verificar si el archivo de base de datos realmente existe en el contenedor, accede a él con:

In [ ]:
docker exec -it sqlite bash
docker exec -it sqlite sh

Esto debería abrir una shell dentro del contenedor. Luego, podemos verificar si el archivo db.sqlite está presente en el directorio /var/lib/sqlite con el siguiente comando:

In [ ]:
cd /var/lib/sqlite
ls

/var/lib/sqlite # ls
sqlite.db # Esta es la base de datos que se debería de haber creado y no ha sido así

Si el archivo NO está allí, es posible que no se haya creado correctamente o que el volumen no esté montado como esperamos; pero SI está dende debería de estar.-

- 2. Verifica el comando SQLite3:

Si el archivo sqlite.db está presente en la ubicación correcta, intentamos abrir la base de datos dentro del contenedor:

In [ ]:
sqlite3 /var/lib/sqlite/sqlite.db

sqlite3 sqlite.db

/var/lib/sqlite # sqlite3 sqlite.db
SQLite version 3.48.0 2025-01-14 11:05:00
Enter ".help" for usage hints.
sqlite> 

Ahora que hmos accedido correctamente a la base de datos SQLite dentro del contenedor, ya puedes ejecutar consultas.

In [ ]:
sqlite> .tables

Esto te mostrará una lista de todas las tablas que existen en la base de datos db.sqlite.

***
***


##  Servicio Backend (Node.js API):

- ### Test: Conexión con SQLite.
    - Comando:
    ```yaml
    curl http://localhost:3000/api/test_db
    ```

    - Nos devuelve un ERROR:
    ```yaml
    <!DOCTYPE html>
    <html lang="en">
    <head>
    <meta charset="utf-8">
    <title>Error</title>
    </head>
    <body>
    <pre>Cannot GET /api/test_db</pre>
    </body>
    </html>
    ```

Parece que el error está relacionado con que la ruta /api/test_db no está definida en nuestro backend de Node.js.

La respuesta que obtenemos indica que el servidor no reconoce esa ruta y devuelve un error `Cannot GET /api/test_db`,
lo que significa que no encuentra una ruta válida para manejar esa solicitud

## Pasos para verificar:

- Verificar la ruta en el código de Node.js:

Asegúrarnos de que tenemos definida una ruta en el backend que maneje las solicitudes a /api/test_db.:

In [ ]:
const express = require('express');
const sqlite3 = require('sqlite3').verbose();
const fs = require('fs');
const path = require('path');

const app = express();
const port = 3000;

// Conectar a la base de datos SQLite
const dbPath = path.join(__dirname, 'data', 'sqlite.db');
const db = new sqlite3.Database(dbPath);

// Ejecutar el script de inicialización de la base de datos desde tools/init.sql
const initSQL = fs.readFileSync(path.join(__dirname, '/tools', 'init.sql'), 'utf-8');
db.exec(initSQL, (err) => {
    if (err) {
        console.error('Error al inicializar la base de datos:', err.message);
    } else {
        console.log('Base de datos inicializada correctamente');
    }
});

// Ruta básica para probar el servidor
app.get('/', (req, res) => {
    res.send('¡Hola, mundo desde Node.js!');
});

// Iniciar el servidor
app.listen(port, () => {
    console.log(`Servidor escuchando en http://localhost:${port}`);
});


En nuestro archivo `app.js`, hemos configurado correctamente el servidor Express y la inicialización de la base de datos SQLite. Sin embargo, falta la ruta para manejar `/api/test_db`.

Actiualizamos archivo con esta ruta para interactuar con la base de datos SQLite y hacer una consulta básica.

Actualización de app.js:

In [ ]:
const express = require('express');
const sqlite3 = require('sqlite3').verbose();
const fs = require('fs');
const path = require('path');

const app = express();
const port = 3000;

// Middleware para parsear JSON en solicitudes POST
app.use(express.json());

// Conectar a la base de datos SQLite
const dbPath = path.join(__dirname, 'data', 'sqlite.db');
const db = new sqlite3.Database(dbPath);

// Ejecutar el script de inicialización de la base de datos desde tools/init.sql
const initSQL = fs.readFileSync(path.join(__dirname, '/tools', 'init.sql'), 'utf-8');
db.exec(initSQL, (err) => {
    if (err) {
        console.error('Error al inicializar la base de datos:', err.message);
    } else {
        console.log('Base de datos inicializada correctamente');
    }
});

// Ruta básica para probar el servidor
app.get('/', (req, res) => {
    res.send('¡Hola, mundo desde Node.js!');
});

// Ruta para obtener todos los usuarios
app.get('/api/users', (req, res) => {
    db.all('SELECT * FROM users', [], (err, rows) => {
        if (err) {
            console.error('Error al consultar la tabla users:', err.message);
            res.status(500).send('Error al consultar la tabla users');
            return;
        }
        // Devuelve los usuarios en formato JSON
        res.json(rows);
    });
});

// Ruta para obtener todos los juegos
app.get('/api/games', (req, res) => {
    db.all('SELECT * FROM games', [], (err, rows) => {
        if (err) {
            console.error('Error al consultar la tabla games:', err.message);
            res.status(500).send('Error al consultar la tabla games');
            return;
        }
        // Devuelve los juegos en formato JSON
        res.json(rows);
    });
});

// Ruta POST para crear un nuevo usuario
app.post('/api/create', (req, res) => {
    const { username, email, password } = req.body;
    if (!username || !email || !password) {
        return res.status(400).send('Faltan campos requeridos');
    }
    const query = `INSERT INTO users (username, email, password) VALUES (?, ?, ?)`;
    db.run(query, [username, email, password], function (err) {
        if (err) {
            console.error('Error al insertar usuario:', err.message);
            return res.status(500).send('Error al insertar usuario');
        }
        res.status(201).send(`Usuario creado con ID: ${this.lastID}`);
    });
});

// Ruta para probar la conexión a la base de datos con una consulta de prueba
app.get('/api/test_db', (req, res) => {
    db.get('SELECT 1', [], (err, row) => {
        if (err) {
            console.error('Error al hacer la consulta de prueba:', err.message);
            res.status(500).send('Error al hacer la consulta de prueba');
            return;
        }
        // Respuesta de éxito si la consulta de prueba fue correcta
        res.send('Conexión a la base de datos exitosa');
    });
});

// Iniciar el servidor
app.listen(port, () => {
    console.log(`Servidor escuchando en http://localhost:${port}`);
});

### Explicación de los cambios:
- Ruta /api/users:
    - Consulta todos los usuarios de la tabla users y los devuelve como JSON.
- Ruta /api/games:
    - Consulta todos los registros de la tabla games y los devuelve como JSON.
- Ruta /api/test_db:

- Ejecutamos una consulta de prueba SELECT 1 para verificar que la conexión a la base de datos funciona correctamente.


Próximos pasos:
- Reconstruir el contenedor backend: Después de realizar estos cambios, necesitas reconstruir tu contenedor para aplicar los cambios en el código.

In [ ]:
make logs_service SERVICE=backend

app  | 
app  | > backend@1.0.0 start
app  | > node app.js
app  | 
app  | Servidor escuchando en http://localhost:3000
app  | Base de datos inicializada correctamente

- Probar las rutas:
    - Usar curl para probar las rutas y asegurarnos de que las consultas están funcionando correctamente. Por ejemplo:

- Obtener todos los usuarios:
- Obtener todos los juegos:
- Probar la conexión a la base de datos:

In [ ]:
curl http://localhost:3000/api/users
[]% 

curl http://localhost:3000/api/games
[]%

curl http://localhost:3000/api/test_db

Conexión a la base de datos exitosa%  

***
***


##  Servicio PHP:

- ### Test: Verificar que PHP está en ejecución.
    - Comando:
    ```yaml
    curl http://localhost:8080/
    ```
    - Nos devuelve un error:
    ```yaml
    curl: (56) Recv failure: Connection reset by peer
    ```
El error `curl: (56) Recv failure: Connection reset by peer` puede indicar que algo está fallando con la conexión entre el servidor Node.js y el cliente al intentar acceder al puerto 8080

Hemos solucionado el error creando un volumen de datos persistentes. 

In [ ]:
services:
    php: 
        build: dockers/php/.
        container_name: php
        image: php:8.1-apache
        volumes:
            - php_data:/var/www/html
        ports:
            - "8080:80"
        networks:
            - transcendence
        depends_on:
            - backend
        restart: always


volumes:
    php_data:
        name: php_data
        driver: local
        driver_opts:
            type: none
            device: "/Users/usuario/data/php"
            o: bind

***
***


## Servicio Frontend (TypeScript + Tailwind):

- ### Test: Verificar que Frontend está en ejecución.<font color="green">**(SOLUCIONADO)**</font>
    - Comando:
    ```yaml
    curl http://localhost:3001/
    ```
    - NO devuelve nada:
    ```yaml
   
    ```
    - Comando:
    ```yaml
    make logs_service SERVICE=frontend
    ````
    - El estado del servicio es perfecto:
    ```yaml
    frontend  | 
    frontend  | > frontend@1.0.0 dev
    frontend  | > vite --host
    frontend  | 
    frontend  | 
    frontend  |   vite v2.9.18 dev server running at:
    frontend  | 
    frontend  |   > Local:    http://localhost:3001/
    frontend  |   > Network:  http://172.18.0.7:3001/
    frontend  | 
    frontend  |   ready in 728ms.
    frontend  | 
    ```


- ### Test: Verificar que Frontend está en ejecución.<font color="green">**(SOLUCIONADO)**</font>
    - Comando:
    ```yaml
    curl http://localhost:3001/
    ```
    - Esperado:
    ```yaml
   <!DOCTYPE html>
    <html lang="en">
    <head>
    <script type="module" src="/@vite/client"></script>

    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Frontend con TypeScript y Tailwind CSS</title>
    <link href="/dist/output.css" rel="stylesheet">
    </head>
    <body class="bg-gray-100">
    <h1 class="text-4xl font-bold text-center mt-10">¡Hola, mundo!</h1>
    <script type="module" src="/src/index.ts"></script>
    </body>
    </html>
    ```
- ## <font color="green">**SOLUCIÓN**</font>
 - `Vite` estaba sirviendo el contenido de manera incorrecta, debido a la ubicación del archivo `index.html`. 
 1. Ubicación correcta de index.html
El archivo `index.html` debe estar en la raíz del proyecto y no dentro de `public/`. Si el archivo index.html está dentro de public/, Vite lo tratará como un archivo estático. Para que Vite lo sirva desde la raíz cuando se ejecuta el comando `npm run dev`, debe estar ubicado directamente en la raíz del proyecto (es decir, al mismo nivel que el directorio src y otros archivos de configuración).


## Servicio Frontend (TypeScript + Tailwind) PRODUCCIÓN:

- ### Acceder a la web desde el PHP.<font color="green">**(SOLUCIONADO)**</font>
    - Error al levantar los contenedores:
    ```yaml
    => ERROR [frontend 6/6] RUN npm run build
    ------
    > [frontend 6/6] RUN npm run build:
    0.876 
    0.876 > frontend@1.0.0 build
    0.876 > vite build
    0.876 
    1.324 vite v2.9.18 building for production...
    1.461 transforming...
    1.500 ✓ 2 modules transformed.
    1.501 Could not resolve './assets/styles.css' from index.html
    1.504 error during build:
    1.504 Error: Could not resolve './assets/styles.css' from index.html
    1.504     at error (/usr/src/app/node_modules/rollup/dist/shared/rollup.js:198:30)
    1.504     at ModuleLoader.handleResolveId (/usr/src/app/node_modules/rollup/dist/shared/rollup.js:22464:24)
    1.504     at /usr/src/app/node_modules/rollup/dist/shared/rollup.js:22427:26
    ------
    failed to solve: process "/bin/sh -c npm run build" did not complete successfully: exit code: 1
    make: *** [all] Error 17
    ```
- #### El error indica que Vite no puede resolver la ruta al archivo ./assets/styles.css durante la construcción. 

Esto puede suceder si el archivo CSS generado no está siendo creado correctamente en la carpeta assets o si la ruta en index.html no coincide con la ubicación del archivo CSS generado por Tailwind.




## <font color="gree">Solución</font> al error de configuración en PHP y Frontend
Este notebook resume los pasos que hemos seguido para solucionar el error en la configuración, donde PHP no servía los archivos estáticos generados por el frontend.

### **Problema inicial**
  - **Frontend**: Generaba archivos estáticos en la carpeta dist dentro del contenedor, pero no los compartía con PHP.
  - **PHP**: Usaba el volumen php_data para servir archivos desde /var/www/html, pero no tenía acceso a los archivos generados por el frontend.
  - **Resultado**: Al acceder a http://localhost:8080, se mostraba la página de información de PHP (phpinfo()) en lugar de la aplicación web.
Solución
Para solucionar este problema, seguimos los siguientes pasos:

## 1. Compartir los archivos estáticos con PHP
Modificamos el archivo docker-compose.yml para que el frontend genere los archivos estáticos directamente en el volumen php_data, que también es utilizado por PHP.

Cambios en docker-compose.yml:
```yaml
services:
  frontend:
    build: dockers/frontend/.
    container_name: frontend
    image: node:18-alpine
    volumes:
      - frontend_data:/usr/src/app  # Volumen para el código fuente
      - php_data:/usr/src/app/dist   # Comparte los archivos estáticos con PHP
    command: sh -c "npm run build && npx serve dist -l 3001"  # Construye y sirve los archivos estáticos
    ports:
      - "3001:3001"
    networks:
      - transcendence
    restart: unless-stopped
    depends_on:
      - backend  # El frontend depende del backend (si necesita la API)

  php:
    build: dockers/php/.
    container_name: php
    image: php:8.1-apache
    volumes:
      - php_data:/var/www/html  # Sirve los archivos estáticos desde el volumen compartido
    ports:
      - "8080:80"
    networks:
      - transcendence
    depends_on:
      - frontend  # PHP depende del frontend (para los archivos estáticos)
    restart: always
```

- Abrimos http://localhost:8080 en el navegador. Deberíamos ver la página generada por el frontend.
- Acceso al frontend directamente: Abrimos http://localhost:3001 en el navegador. Esto seguirá funcionando, pero ahora PHP también sirve los mismos archivos.

## Explicaciones técnicas
  - **Volúmenes en Docker**:
  Los volúmenes en Docker permiten persistir datos y compartirlos entre contenedores. En este caso, usamos el volumen php_data para compartir los archivos estáticos generados por el frontend con PHP.

  - **Dependencias entre servicios**:
  El parámetro depends_on en Docker Compose se usa para definir dependencias entre servicios. Esto asegura que un servicio espere a que otro servicio esté listo antes de iniciarse. En este caso, PHP depende del frontend para asegurarse de que los archivos estáticos estén listos antes de servir.

  - **Servidores de archivos estáticos**:
  Vite es una herramienta de construcción y servidor de desarrollo para aplicaciones modernas de JavaScript y TypeScript. serve es un servidor de archivos estáticos muy sencillo. En este caso, usamos serve para servir los archivos estáticos generados por Vite.

## Conclusión
Con estos cambios, PHP debería servir los archivos estáticos generados por el frontend en http://localhost:8080. Esto unifica el acceso a la aplicación a través de un solo puerto (8080) en lugar de tener que acceder a través de dos puertos diferentes (3001 y 8080).


## **Test: Comunicación con Backend** <font color="green">(SOLUCIONADO)</font>

 - Acción: Asegurarse de que la aplicación frontend puede realizar peticiones al backend.
 - Resultado esperado: El frontend debe mostrar los datos que provienen de la API de backend (por ejemplo, una lista de ítems).

1. Problema actual
- **Backend**: Está funcionando correctamente y escuchando en http://localhost:3000.

- **Frontend**: No está recibiendo datos del backend cuando intenta acceder a /api/items.

- **Resultado de curl**: Al hacer curl http://localhost:3000/api/items, obtenemos una respuesta vacía ([]).

2. Análisis del problema

- El problema radica en la ruta /api/items del backend. Actualmente, esta ruta está consultando la tabla users en lugar de una tabla items. Esto explica por qué obtienes una respuesta vacía: la tabla users no tiene datos relevantes para esta consulta, o no es la tabla correcta.


- **Paso 1**: Corregir la consulta SQL.
    - En el archivo app.js, la ruta /api/items está consultando la tabla users.
- **Paso 2**: Cambiar la consulta:
    - No tenemos una tabla items en `init.sql`, la creamos y cambia la consulta para que seleccione datos de esa tabla:
- **Paso 3**: Configurar CORS en el backend y Verificación del backend.
    - Aseguramos que el backend está operativo y manejando las solicitudes en http://localhost:3000/api/items, pero se detectó que el endpoint devolvía un array vacío.
    - Para permitir que el frontend realice solicitudes al backend, asegúrate de que el backend esté configurado para manejar solicitudes CORS.
- **PAso 4**: Verificación del frontend.
    - Revisamos que el frontend estuviera construido correctamente y que la comunicación con el backend se estableciera correctamente. Además, comprobamos que la aplicación estuviera sirviendo la web a través de Vite.

Asegurarse de que haya datos en la base de datos que el backend pueda devolver en la ruta /api/items. <font color="green"> (**SOLUCINADO**)</font>
Revisar la lógica de la API en el backend para asegurar que los datos se están devolviendo correctamente.
Verificar la comunicación entre el frontend y el backend, y ajustar las rutas o el método de conexión si es necesario.
Este es un resumen breve que puedes comentar a lo largo de tu proyecto para documentar cómo se abordó el problema actual.

# Solución de error en la comunicación Backend - SQLite

## Descripción del problema

Teníamos un error en la comunicación entre el backend (Node.js) y la base de datos SQLite. El backend no podía crear nuevos ítems en la base de datos, lo que se manifestaba con el error "Cannot POST /api/items" al intentar enviar una solicitud POST a la API.

Además, el contenedor `sqlite` mostraba un error en sus logs: `/var/lib/sqlite/init_db.sh: line 8: can't open /var/lib/sqlite/init.sql: no such file`. Sin embargo, este error era engañoso (un "falso positivo"), ya que la base de datos se creaba correctamente gracias al backend.

## Diagnóstico

1. **Verificación de logs:**
   - Los logs del backend mostraban el mensaje "Conectado a la base de datos SQLite", lo que indicaba que la conexión inicial era exitosa.
   - Los logs del contenedor `sqlite` mostraban el error mencionado anteriormente.

2. **Inspección del contenedor `sqlite`:**
   - Ejecutamos `docker exec -it sqlite sh` para acceder al contenedor `sqlite`.
   - Dentro del contenedor, ejecutamos `sqlite3 sqlite.db` y luego `SELECT * FROM items;` para verificar el contenido de la base de datos.
   - Observamos que los datos iniciales (insertados desde `init.sql`) estaban presentes, lo que confirmaba que la base de datos se había creado correctamente.

3. **Análisis del Dockerfile del backend:**
   - Observamos que el Dockerfile del backend copiaba el archivo `init.sql` a la ruta `/var/lib/sqlite` dentro del contenedor backend: `COPY tools/init.sql /var/lib/sqlite/init.sql`.
   - Esto explicaba por qué el backend podía crear la base de datos, pero también por qué el contenedor `sqlite` no encontraba `init.sql` en su propio sistema de archivos.

## Solución

1. **Implementación del endpoint POST en el backend:**
   - Implementamos la lógica necesaria en el backend (Node.js con Express) para manejar las solicitudes POST a `/api/items`.
   - Esto incluyó:
     - Uso de `body-parser` para parsear el cuerpo de las solicitudes JSON.
     - Validación de los datos recibidos (`name` y `description`).
     - Ejecución de la consulta SQL `INSERT` para crear el nuevo ítem.
     - Envío de una respuesta JSON con el nuevo ítem y su ID.

2. **Modificación del script `init_db.sh`:**
   - Modificamos el script `init_db.sh` en el contenedor `sqlite` para que solo intente inicializar la base de datos si el archivo `sqlite.db` no existe:

     ```sh
     if [ ! -f /var/lib/sqlite/sqlite.db ]; then
         # ... (código para crear la base de datos)
     fi
     ```

   - Esto evita el error "falso positivo" en los logs del contenedor `sqlite`.

## Tests

1. **Test de conexión:**
   - Verificamos los logs del backend para confirmar el mensaje "Conectado a la base de datos SQLite".

2. **Test de creación de ítems:**
   - Enviamos una solicitud POST a `/api/items` con datos válidos (`name` y `description`).
   - Verificamos que la respuesta contenga el nuevo ítem con un ID generado.
   - Consultamos la base de datos directamente (usando `docker exec` y `sqlite3`) para confirmar que el nuevo ítem se ha creado.

3. **Test de persistencia:**
   - Creamos varios ítems usando el endpoint POST.
   - Reiniciamos los contenedores (`docker-compose down && docker-compose up -d`).
   - Consultamos la base de datos (directamente o a través de la API) para verificar que los ítems persisten después del reinicio.

4. **Test de `init_db.sh`:**
   - Verificamos los logs del contenedor `sqlite` para confirmar que ya no aparece el error relacionado con `init.sql`.
   - Esto asegura que el script `init_db.sh` está funcionando correctamente y evitando la reinicialización de la base de datos.

## Conclusión

Con estos cambios y tests, hemos solucionado el problema de comunicación entre el backend y SQLite, y hemos asegurado que los datos se persisten correctamente en el volumen compartido. El error en los logs del contenedor `sqlite` ha sido resuelto y ya no interfiere con el funcionamiento de la aplicación.

***
***

## Servicio Blockchain (avalanche):

**Test: Verificar que avalache está levantado** <font color="green">**(SOLUCIONADO)**</font>

 - Comando:
    ```yaml
    make ps
    ```
 - Servicio no está levatado:
    ```yaml
    Restarting (1) 18 seconds ago       
    ```
- Buscamos los posibles errores con el Comando:
   ```yaml
   make logs_service SERVICE=avalanche
   ````
- Devuelve este error:
   ```yaml
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   ```
- El error `"exec /avalanchego/avalanchego: no such file or directory"` indica que el binario de AvalancheGo no se encuentra en la ruta especificada (/avalanchego/avalanchego) dentro del contenedor 

In [ ]:
 docker exec -it blockchain sh  

- Listamos que hay dentro del contenedor y nos devuleve este error:
```yaml
Error response from daemon: Container b5ea0a2e15b059f91b3666984e10ffa47cab5e7eb890abd936bb1b4c9ba3c8e5 is restarting, wait until the container is running
 ```

No podemos entrar en el contenedor porque no está levantado

- ## SOLUCIÓN:

    - **Añadir glibc a Alpine:** Agregar `glibc` al contenedor Alpine para proporcionar las bibliotecas que faltan.
    ```yaml
    RUN apk add --no-cache libc6-compat
    ```
    - **Verificar la variable AVALANCHEGO_DATA_DIR:** El binario avalanchego estaba buscando archivos de configuración o datos en una ruta específica definida por la variable de entorno `AVALANCHEGO_DATA_DIR`.  Configuramos correctamente en el docker-compose.yml.
    ```yaml
    services:
        avalanche:
            environment:
            - AVALANCHEGO_DATA_DIR=/avalanchego
    ```

**Test: Verificar que avalache está en ejecución** <font color="red">**(ERROR POR SOLUCIONAR)**</font>

In [ ]:
curl http://localhost:9650/ext/health

### Resultado esperado: 
- Un JSON con el estado healthy: true si el nodo está funcionando correctamente.
### RECIBIDO:
```yaml
curl: (56) Recv failure: Connection reset by peer
```

### IMPORTANTE:
1. Advertencia sobre UPnP o NAT-PMP

En los logs, ves el siguiente mensaje:
```yaml
UPnP or NAT-PMP router attach failed, you may not be listening publicly. Please confirm the settings in your router
```

Este mensaje indica que el nodo no pudo adjuntarse al enrutador utilizando UPnP o NAT-PMP, lo que puede afectar la posibilidad de que tu nodo sea accesible públicamente. Esto no es un error crítico si no planeas que tu nodo sea públicamente accesible desde fuera de tu red local. Sin embargo, si es necesario acceder al nodo desde fuera, verifica las configuraciones del enrutador y permite que se redirijan los puertos 9650-9651 a tu contenedor.

## Acciones a seguir:

- Verificar los puertos: 
    - Asegúrate de que los puertos 9650 y 9651 estén abiertos en tu router y redirigidos correctamente.
- Desactivar NAT Traversal: 
    - Si no necesitas la accesibilidad pública, puedes desactivar la opción attemptedNATTraversal en el archivo de configuración de Avalanche.
2. Conexión Bootstrap

La configuración del nodo intenta conectarse a varios nodos bootstrap para unirse a la red de Avalanche. Si el nodo no puede establecer esas conexiones, puede causar problemas con la sincronización del estado del blockchain. Aquí están los nodos bootstrap configurados en tu caso:

```yaml
"bootstrapIDs":["NodeID-kZNuQMHhydefgnwjYX1fhHMpRNAs9my1","NodeID-QKGoUvqcgormCoMj6yPw9isY7DX9H4mdd","NodeID-FGRoKnyYKFWYFMb6Xbocf4hKuyCBENgWM","NodeID-HsBEx3L71EHWSXaE6gvk2VsNntFEZsxqc","NodeID-A7GwTSd47AcDVqpTVj7YtxtjHREM33EJw"],
"bootstrapIPs":[{"ip":"18.158.15.12","port":9651},{"ip":"18.162.161.230","port":9651},{"ip":"122.248.200.212","port":9651},{"ip":"3.106.25.139","port":9651},{"ip":"3.21.38.33","port":9651}]
```
Si estos nodos no están disponibles o no se pueden alcanzar, deberías intentar conectarte a otros nodos bootstrap. Puedes intentar modificar el archivo de configuración de tu nodo Avalanche y probar con diferentes nodos bootstrap.

## Siguientes pasos:
Confirmar los ajustes de la red y la conectividad del nodo, asegurando que los puertos estén abiertos y redirigidos correctamente.
Verificar el acceso a los nodos bootstrap configurados. Si no logras conectarte, puedes buscar otros nodos bootstrap públicos y actualizarlos en tu archivo de configuración.

- Hemos creado un archivo config.json:
```yaml
{
    "network-id": "local",
    "http-host": "0.0.0.0",
    "http-port": 9650,
    "staking-enabled": false,
    "bootstrap-ips": "18.158.15.12:9651,18.162.161.230:9651,122.248.200.212:9651,3.106.25.139:9651,3.21.38.33:9651",
    "bootstrap-ids": "NodeID-kZNuQMHhydefgnwjYX1fhHMpRNAs9my1,NodeID-QKGoUvqcgormCoMj6yPw9isY7DX9H4mdd,NodeID-FGRoKnyYKFWYFMb6Xbocf4hKuyCBENgWM,NodeID-HsBEx3L71EHWSXaE6gvk2VsNntFEZsxqc,NodeID-A7GwTSd47AcDVqpTVj7YtxtjHREM33EJw",
    "db-dir": "/root/.avalanchego/db",
    "log-dir": "/root/.avalanchego/logs",
    "log-level": "info",
    "ipConfig": {
        "ip": {
            "ip": "172.18.0.3",
            "port": 9651
        },
        "ipResolutionFrequency": 30000000000
    }
  }
```
### El error ha cambiado:


In [ ]:
curl http://localhost:9650/ext/health   

```yaml
{"checks":{"P":{"error":"not yet run","timestamp":"0001-01-01T00:00:00Z","duration":0},"bootstrapped":{"error":"not yet run","timestamp":"0001-01-01T00:00:00Z","duration":0},"database":{"timestamp":"2025-02-15T22:05:35.665058613Z","duration":2194},"diskspace":{"message":{"availableDiskBytes":4109177454592},"timestamp":"2025-02-15T22:05:35.665053751Z","duration":34274},"network":{"message":{"connectedPeers":0,"sendFailRate":0,"timeSinceLastMsgReceived":"483238h5m35.665497391s","timeSinceLastMsgSent":"483238h5m35.665497391s"},"error":"network layer is unhealthy reason: not connected to a minimum of 1 peer(s) only 0, no messages from network received in 483238h5m35.665497391s \u003e 1m0s, no messages from network sent in 483238h5m35.665497391s \u003e 1m0s","timestamp":"2025-02-15T22:05:35.665535675Z","duration":539419,"contiguousFailures":1,"timeOfFirstFailure":"2025-02-15T22:05:35.665535675Z"},"router":{"message":{"longestRunningRequest":"0s","outstandingRequests":0},"timestamp":"2025-02-15T22:05:35.664993721Z","duration":19539}},"healthy":false}
```

## Test: Comunicación con Backendn <font color="red">**(ERROR)**</font>
        
- Comando:

In [ ]:
curl http://localhost:3000/api/avalanche_status

### Acción:
- Crear un endpoint en el backend que interactúe con Avalanche
### Resultado esperado: 
- El backend debería devolver información relevante del estado de la blockchain.
### <font color="red">Resultado RECIBIDO</font> : 
```yaml
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot GET /api/avalanche_status</pre>
</body>
</html>
```

***
***


## Servicio OWASP ZAP (Seguridad):

## **Test: Verificar que OWASP ZAP está en ejecución** <font color="green">**(SOLUCIONADO)**</font>

- Comando:

```yaml
curl http://localhost:8081/
```
- Acción:

    - Realizar una petición GET a la interfaz de OWASP ZAP.
- Resultado esperado:

    - La interfaz de ZAP debería cargar, indicando que está funcionando.

- <font color="red">**RECIBIDO:**</font>
```yaml
curl: (52) Empty reply from server
```

## **Test: Escaneo de vulnerabilidades y MOSTRAR EN EL NAVEGADOR**:

- Acción: Ejecutar un escaneo desde OWASP ZAP contra el backend y el frontend.
- Resultado esperado: Un reporte con posibles vulnerabilidades de seguridad.

¡Tienes razón! Vamos a recapitular todo el proceso desde el principio, enfocándonos en el problema original de ZAP y cómo lo resolvimos.

**Problema Inicial:**

* **ZAP y Reportes HTML:**
    * El contenedor `security` utiliza OWASP ZAP para realizar escaneos de seguridad.
    * ZAP genera reportes en formato HTML, pero no está configurado para servirlos directamente a través de un navegador.
    * Por lo tanto, necesitábamos una forma de acceder a estos reportes HTML desde un navegador.
* **Necesidad de Servir los Reportes:**
    * Decidimos utilizar el contenedor `php` (con Apache) para servir los reportes HTML generados por ZAP.
    * Esto implicaba copiar los reportes desde el contenedor `security` al contenedor `php`.

**Desarrollo y Soluciones:**

1.  **Volumen Compartido:**
    * Para evitar la necesidad de copiar manualmente los archivos, implementamos un volumen compartido (`zap_reports`) entre los contenedores `security` y `php`.
    * Esto permitió que ambos contenedores accedieran al mismo directorio en el sistema de archivos.

2.  **Problemas con la Copia y el Acceso:**
    * Inicialmente, tuvimos dificultades para asegurarnos de que el directorio donde se guardaban los reportes existiera en el contenedor `php` y que Apache pudiera acceder a los archivos.
    * También enfrentamos problemas con los permisos de los archivos y la configuración de Apache.

3.  **Solución Final:**
    * **Configuración del Alias de Apache:**
        * Corregimos la configuración del alias en `zap_reports.conf` para que apuntara a la ruta correcta dentro del volumen compartido: `Alias /zap_reports /zap/reports`.
        * Esto soluciono el problema de que el navegador no encontraba el archivo.
    * **Enlace Simbólico:**
        * Se configuro un enlace simbolico para que la carpeta donde apache sirve los archivos apuntara a la carpeta del volumen compartido.
    * **Automatización:**
        * Se automatizo la creación del directorio y los permisos dentro del contenedor php.
    * **Contenedor auxiliar:**
        * Se creo un contenedor auxiliar para poder ejecutar los comandos de docker dentro del contenedor security, ya que este no tiene instalado docker.

**Pruebas para Verificar la Solución:**

1.  **Verificar la Existencia del Archivo:**
    * Ejecuta `docker exec -it php sh` y luego `ls /zap/reports` para confirmar que el archivo `zap_report.html` está presente.
2.  **Verificar el Enlace Simbólico:**
    * Ejecuta `docker exec -it php sh` y luego `ls -l /var/www/html/zap_reports` para verificar que el enlace simbólico apunta correctamente.
3.  **Verificar la Configuración de Apache:**
    * Ejecuta `docker logs php` para asegurarte de que no haya errores.
4.  **Acceder al Reporte desde el Navegador:**
    * Abre tu navegador y accede a `http://localhost:8080/zap_reports/zap_report.html`.
5.  **Verificar los permisos del archivo:**
    * Ejecuta `docker exec -it php sh` y verifica los permisos del archivo con `ls -l /var/www/html/zap_reports/zap_report.html`. Si los permisos son incorrectos, cambia los permisos con `chmod 644 /var/www/html/zap_reports/zap_report.html`.
6.  **Crear un archivo HTML de prueba:**
    * `docker exec -it php sh`, `echo "<h1>Test</h1>" > /var/www/html/zap_reports/test.html`, y acceder a `http://localhost:8080/zap_reports/test.html`.



# COMANDO SCAN ZAP:

In [ ]:
docker exec -it security /zap/wrk/zap_scan.sh

# MOSTRAR ARCHIVO DE ESCANEO:

In [ ]:
http://localhost:8080/zap_reports/zap_report.html

### Utiliza una herramienta de inspección de red: <font color="red">ERROR</font>
- Utiliza herramientas como tcpdump o wireshark para capturar el tráfico de red y analizar las conexiones entre el host y el contenedor security. Esto te puede ayudar a identificar problemas de conectividad o de configuración de la red.

In [ ]:
docker exec -it security tcpdump -i eth0 -n -X port 8081

```yaml
OCI runtime exec failed: exec failed: unable to start container process: exec: "tcpdump": executable file not found in $PATH: unknown
```

### Indica que el comando tcpdump no se encuentra en la ruta de ejecución ($PATH) dentro del contenedor security.

### Solución:

- Instalar tcpdump dentro del contenedor: